# Локальный CPU: ddddocr и PP-OCRv6

In [ ]:
from pathlib import Path
import sys, random
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'captcha_lab').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
random.seed(42)
import numpy as np
import pandas as pd
np.random.seed(42)
from captcha_lab.data import image_manifest, ensure_labels, crop_image, save_json
from captcha_lab.benchmark import run_benchmark, save_run, evaluate
print('Python:', sys.version.split()[0])

Гипотеза: готовая специализированная OCR может читать короткую строку без большой VLM.
Сравниваем ddddocr default/beta и PP-OCRv6 tiny/small. Первая загрузка может скачать веса.
Таймер исключает загрузку модели; первый inference записан отдельно. Три повтора на изображение —
предварительный замер. Confidence разных моделей не является общей шкалой точности.

In [ ]:
from captcha_lab.onnx_models import load_ddddocr, load_ppocr
manifest = image_manifest(); assert not manifest.empty
CONFIG = dict(repeats=3, variants=('rgb','gray'))
factories = {'ddddocr-default':lambda:load_ddddocr(False),'ddddocr-beta':lambda:load_ddddocr(True),
             'ppocrv6-tiny':lambda:load_ppocr('tiny'),'ppocrv6-small':lambda:load_ppocr('small')}

In [ ]:
frames, failures = [], {}
for name, factory in factories.items():
    print('Модель:',name,flush=True)
    try:
        recognizer = factory()
        frame = run_benchmark(manifest,recognizer,name,**CONFIG)
        save_run(frame,name+'.csv'); frames.append(frame)
        del recognizer
    except Exception as exc:
        failures[name] = type(exc).__name__
save_json(ROOT/'results/onnx_failures.json',failures)
print('Ошибки загрузки/запуска:',failures)
assert frames, 'Ни одна модель не запустилась; проверьте зависимости/доступ к весам'
predictions = pd.concat(frames,ignore_index=True)
display(predictions[['model','image_id','variant','prediction','median_ms','error']])

In [ ]:
labels = ensure_labels(manifest)
display(evaluate(predictions,labels))
print('Если verified=0, exact_match/CER отсутствуют: это не нулевая и не высокая точность.')